# Synthetic Data Generation with Ollama

This notebook connects to a local Ollama instance to discover available models and generate synthetic training data for the Lite-NLP Workbench.

In [ ]:
import requests
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import json
import os

OLLAMA_BASE_URL = 'http://localhost:11434'

def get_ollama_models():
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
        response.raise_for_status()
        models = response.json().get('models', [])
        return [model['name'] for model in models]
    except Exception as e:
        print(f"Error connecting to Ollama: {e}\nMake sure Ollama is running.")
        return []

available_models = get_ollama_models()

In [ ]:
if available_models:
    model_dropdown = widgets.Dropdown(
        options=available_models,
        description='Ollama Model:',
        disabled=False,
    )
    display(model_dropdown)
else:
    print("No models available to select. Please pull a model using `ollama run llama3` or similar.")

In [ ]:
def generate_samples(model_name, intent_name, inclusion_goals, exclusion_goals, num_samples=10):
    samples = []
    
    # Generate Good Samples
    good_prompt = f"""Generate {num_samples} distinct examples of text for a {intent_name}.
Requirements (Include): {inclusion_goals}
Do NOT include: {exclusion_goals}
Output ONLY a valid JSON array of strings containing the examples, without any markdown formatting or extra text."""
    
    # Generate Bad Samples
    bad_prompt = f"""Generate {num_samples} distinct examples of text for a {intent_name} that blatantly VIOLATE the rules.
The text MUST INCLUDE: {exclusion_goals}
Output ONLY a valid JSON array of strings containing the examples, without any markdown formatting or extra text."""

    for label, prompt in [('Good', good_prompt), ('Bad', bad_prompt)]:
        print(f"Generating {label} samples for {intent_name}...")
        try:
            response = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json={
                'model': model_name,
                'prompt': prompt,
                'stream': False,
                'format': 'json'
            })
            response.raise_for_status()
            
            result_text = response.json().get('response', '[]')
            generated_texts = json.loads(result_text)
            
            for text in generated_texts:
                samples.append({'text': text, 'label': label, 'intent': intent_name})
        except json.JSONDecodeError:
            print(f"Failed to parse JSON response for {label} samples.")
        except Exception as e:
            print(f"Error generating {label} samples: {e}")
            
    return samples

In [ ]:
if 'model_dropdown' in locals():
    selected_model = model_dropdown.value
    print(f"Using model: {selected_model}")

    # Define Intents based on requirements
    intents = [
        {
            'name': 'Biographical',
            'inclusion_goals': 'Physical traits, history, personality.',
            'exclusion_goals': 'Addresses, Social Security Numbers (SSNs), medical history (PHI).'
        },
        {
            'name': 'Incident Report',
            'inclusion_goals': 'Dates, locations, objective actions.',
            'exclusion_goals': 'Emotional rants, hearsay, identifying victim names.'
        },
        {
            'name': 'Product Review',
            'inclusion_goals': 'Quality, usability, value.',
            'exclusion_goals': 'Shipping complaints, personal attacks, promo links.'
        },
        {
            'name': 'Technical Support',
            'inclusion_goals': 'Error codes, steps taken, hardware model.',
            'exclusion_goals': 'Passwords, credit card numbers, unrelated small talk.'
        }
    ]

    all_data = []
    for intent in intents:
        samples = generate_samples(
            model_name=selected_model,
            intent_name=intent['name'],
            inclusion_goals=intent['inclusion_goals'],
            exclusion_goals=intent['exclusion_goals'],
            num_samples=10  # Generating 10 good and 10 bad samples per intent
        )
        all_data.extend(samples)

    # Save Data
    if all_data:
        df = pd.DataFrame(all_data)
        os.makedirs('../data', exist_ok=True)
        output_path = '../data/intent_library.csv'
        df.to_csv(output_path, index=False)
        print(f"\nData generation complete. Saved {len(df)} rows to {output_path}")
        display(df.head())
        display(df['intent'].value_counts())
    else:
        print("No data generated.")